In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import pickle
import matplotlib.pyplot as plt

from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import StandardScaler

In [2]:
main_df = pd.read_csv(f"small_data/main_df.csv")

In [3]:
main_df

,Exp_ID,Image_Metadata_Site,track_id,Image_Metadata_T,Nuclear_size,ERKKTR_ratio,FoxO3A_ratio,objNuclei_Location_Center_X,objNuclei_Location_Center_Y
0,1,1,1,0,303.000,0.704407,1.33383,932.211,875.248
1,1,1,1,1,333.000,0.848242,1.33791,932.150,874.174
2,1,1,1,2,314.000,1.059170,1.37627,932.376,873.787
3,1,1,1,3,322.000,1.188000,1.35754,932.168,873.453
4,1,1,1,4,313.999,1.205540,1.36695,931.146,872.885
...,...,...,...,...,...,...,...,...,...
356627,1,1,2857,253,165.000,0.597341,1.32403,1019.050,771.624
356628,1,1,2857,254,192.000,0.587024,1.28853,1019.060,773.099
356629,1,1,2857,255,200.000,0.587822,1.28325,1019.100,773.540
356630,1,1,2857,256,192.000,0.579069,1.29542,1018.930,772.469


In [4]:
sorted_main_df = main_df.set_index(['Image_Metadata_T', 'track_id']).sort_index()
sorted_main_df

Exp_ID  Image_Metadata_Site  Nuclear_size  \
Image_Metadata_T track_id                                              
0                1              1                    1       303.000   
                 2              1                    1       268.001   
                 3              1                    1       370.000   
                 4              1                    1       341.000   
                 5              1                    1       308.000   
...                           ...                  ...           ...   
257              2849           1                    1       165.000   
                 2850           1                    1       138.000   
                 2855           1                    1       148.000   
                 2856           1                    1       148.000   
                 2857           1                    1       175.000   

                           ERKKTR_ratio  FoxO3A_ratio  \
Image_Metadata_T track_id                               
0                1             0.704407       1.33383   
                 2             1.227760       1.20592   
                 3             0.779226       1.27561   
                 4             0.870780       1.34460   
                 5             0.807262       1.47321   
...                                 ...           ...   
257              2849          0.607956       1.31242   
                 2850          0.775133       1.15722   
                 2855          1.114600       1.25315   
                 2856          0.861735       1.16059   
                 2857          0.589269       1.31594   

                           objNuclei_Location_Center_X  \
Image_Metadata_T track_id                                
0                1                             932.211   
                 2                             162.328   
                 3                             647.816   
                 4                             642.988   
                 5                             990.718   
...                                                ...   
257              2849                          839.988   
                 2850                         1019.510   
                 2855                          411.797   
                 2856                         1018.580   
                 2857                         1018.850   

                           objNuclei_Location_Center_Y  
Image_Metadata_T track_id                               
0                1                            875.2480  
                 2                            365.4140  
                 3                            846.5760  
                 4                            827.8650  
                 5                             59.4675  
...                                                ...  
257              2849                         877.9760  
                 2850                          50.1159  
                 2855                         205.4320  
                 2856                         809.4530  
                 2857                         771.6110  

[356632 rows x 7 columns]

In [5]:
def read_graph(path: str) -> nx.Graph:
    with open(path, "rb") as file:
        loaded_data = pickle.load(file)

    return loaded_data

In [ ]:
graphs: dict[int, nx.Graph]

for time_t in sorted_main_df.index.get_level_values(0).unique():
    print(time_t)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
